# 03 - Data Preprocessing

**Project:** Student Placement & Salary Prediction  
**Goal of this notebook:** data ko model ke layak banana: missing values, duplicates, categorical encoding, aur encoders save karna.

**Notebook order:** 01 Data Understanding -> 02 EDA -> 03 Data Preprocessing -> 04 Feature Engineering -> 05 Model Training -> 06 Model Evaluation. Ye notebook `notebook/` folder ke andar se run karo (paths `../datasets` aur `../models` relative hain).

## 1. Libraries aur data load karna
`LabelEncoder` text columns ko numbers me badalne ke liye, `pickle` encoders save karne ke liye. Ye wahi steps hain jo dashboard ke `utils.py` me `prepare_data()` me hain.

In [ ]:
import os, pickle
import pandas as pd
from sklearn.preprocessing import LabelEncoder

raw = pd.read_csv('../datasets/studentplacement.csv')
df = raw.copy()          # original raw data ko safe rakhte hain
print('Start shape:', df.shape)

## 2. Missing values handle karna
Numeric columns ki missing values ko column ke **mean** se fill karte hain. Notebook 01 me dekha tha ki is dataset me koi missing value nahi hai, isliye is step se data nahi badlega. Phir bhi ye step pipeline me rakha hai taaki nayi/gandi file par bhi code chale.

In [ ]:
print('Missing before:', int(df.isnull().sum().sum()))
df = df.fillna(df.mean(numeric_only=True))
print('Missing after :', int(df.isnull().sum().sum()))

## 3. Duplicate rows hatana
Repeat hui rows model ko galat tarah se bias karti hain, isliye `drop_duplicates()` lagate hain. Is dataset me duplicate nahi mile, to shape wahi rahegi.

In [ ]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print('Rows before:', before, '| after:', len(df), '| removed:', before - len(df))

## 4. Categorical columns ko encode karna (Label Encoding)
ML models text nahi samajhte, isliye har text column ko `LabelEncoder` se numbers me badalte hain (categories alphabetical order me 0, 1, 2, ... milte hain). Har column ka alag encoder `encoders` dictionary me rakhte hain.

In [ ]:
encoders = {}
for col in df.select_dtypes(include=['object', 'string']).columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

for col, le in encoders.items():
    print(col, '->', {c: int(i) for i, c in enumerate(le.classes_)})

## 5. Check: ab koi text column bacha?
Preprocessing ke baad saare columns numeric hone chahiye.

In [ ]:
print('Text columns left:', df.select_dtypes(include=['object', 'string']).columns.tolist())
df.dtypes.value_counts()

## 6. Encoders ko `models/` folder me save karna
Prediction ke time (dashboard) wahi encoders chahiye jo training me use hue, nahi to `Male` ka number alag ban jayega. Isliye `models/encoders.pkl` me save karte hain. Dashboard ka Bulk Prediction page isi file ko load karta hai.

In [ ]:
os.makedirs('../models', exist_ok=True)
with open('../models/encoders.pkl', 'wb') as f:
    pickle.dump(encoders, f)
print('Saved:', os.path.abspath('../models/encoders.pkl'))

## 7. Processed data ko `datasets/processed/` me save karna
Encoded data ko CSV me save karte hain, taaki agla notebook (feature engineering) seedha isse load kare.

In [ ]:
os.makedirs('../datasets/processed', exist_ok=True)
df.to_csv('../datasets/processed/encoded_data.csv', index=False)
print('Saved encoded_data.csv with shape', df.shape)
df.head()

## 8. Summary

- Missing values: 0, duplicates: 0, isliye rows 100,000 hi rahi.
- 5 text columns (`gender`, `branch`, `college_tier`, `volunteer_experience`, `placement_status`) numbers me encode hue.
- `models/encoders.pkl` aur `datasets/processed/encoded_data.csv` ban gaye.
- **Scaler nahi use kiya:** is project ke models (Decision Tree, Random Forest, Extra Trees, Gradient Boosting) tree-based hain jo feature scale par depend nahi karte, aur Linear Regression ka (least squares) prediction scaling se nahi badalta. Isliye koi scaler save nahi hua.